# Significance Testing via Permutation Tests

- Text level was shuffled in training w/ a different random seed for each permutation

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import ALPHA_LIST
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from itertools import product
from pathlib import Path
import json
import shlex

In [ ]:
search_dict = {
    "agg": ["0_shot_binary", "0_shot_proba-limited", "avg", "block", "max"],
    "llama_model": ["Llama-3.3-8B-Instruct", "Llama-3.3-70B-Instruct"],
    "desc": ["desc_augment", "desc_augment_neg", "desc_original"],
    "seg": ["pt_noshort", "turns"],
    "alpha": ALPHA_LIST,
    "repeats": [f"repeat_{i}" for i in range(10)],
    "splits": [f"split_{i}.parquet" for i in range(3)],
    "split_perms": [f"split_{i}_perm_ids.parquet" for i in range(3)],
}

## Aggregate r^2 values across perms

In [ ]:
def create_cmd(impact_type, num_layers, perm_dir, search_dict, export_path, n_threads):
    cmd_str_list = [
        f"export PYTHONPATH={BASE_PATH};",
        f"export OMP_NUM_THREADS={n_threads};",
        "uv run",
        "python -m src.perm_res",
        "--impact_type",
        str(impact_type),
        "--num_layers",
        str(num_layers),
        "--perm_dir",
        str(perm_dir),
        "--export_path",
        str(export_path),
        "--search_dict",
        shlex.quote(json.dumps(search_dict)),
    ]
    return " ".join(cmd_str_list)

In [ ]:
perm_check_start = 0
perm_check_end = 50  # not inclusive
INCLUDE_BASELINE = True

In [ ]:
full_cmd_list = []
loop_dir = BASE_PATH / "perm_res" / "preds"
swarm_dir = BASE_PATH / "swarm_dir/perm_r2"
cmd_dir = swarm_dir / "commands"
log_dir = swarm_dir / "logs"
export_dir = BASE_PATH / "perm_res" / "r2_tables"
for impact_type in ["parent", "self"]:
    for num_layers in [2, 3]:
        swarm_path = cmd_dir / f"{impact_type}_{num_layers}.swarm"
        swarm_path.parent.mkdir(exist_ok=True, parents=True)
        if swarm_path.exists():
            swarm_path.unlink()
        cmd_list = []
        for perm_dir in loop_dir.iterdir():
            export_path = (
                export_dir / perm_dir.name / f"{impact_type}_{num_layers}.parquet"
            )
            perm_name = perm_dir.name
            if perm_name == "baseline":
                if not INCLUDE_BASELINE:
                    continue
            else:
                perm_num = int(perm_name.split("_")[1].strip(""))
                if (perm_num < perm_check_start) or (perm_num >= perm_check_end):
                    continue
            print(perm_name)
            cmd_str = create_cmd(
                impact_type=impact_type,
                num_layers=num_layers,
                perm_dir=perm_dir,
                search_dict=search_dict,
                n_threads=2,
                export_path=export_path,
            )
            cmd_list.append(cmd_str)
        swarm_path.write_text("\n".join(cmd_list))
        full_cmd_list += cmd_list
print(f"Num commands: {len(full_cmd_list)}")

Run w/ swarm

In [ ]:
from src.swarm import run_perm_r2_swarm

run_perm_r2_swarm(
    impact_list=["parent", "self"],
    layer_list=[2, 3],
    log_dir=log_dir,
    cmd_dir=cmd_dir,
    n_batch_for_3=5,
)

Run sequentially

In [ ]:
# import subprocess

# for iteration, cmd in enumerate(full_cmd_list):
#     print(f"{iteration +1}/{len(full_cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

## Get p-values

In [ ]:
base_imp_path = BASE_PATH / "perm_res" / "r2_tables"
parent_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_2_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_2.parquet")
    for perm_dir in base_imp_path.iterdir()
}
parent_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "parent_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}
self_3_df_dict = {
    perm_dir.name: pd.read_parquet(base_imp_path / perm_dir.name / "self_3.parquet")
    for perm_dir in base_imp_path.iterdir()
}

In [ ]:
def get_p_df(search_dict, num_layers, r2_df_dict):
    p_val_list = []
    for agg in search_dict["agg"]:
        for llama_model in search_dict["llama_model"]:
            if agg in ["max", "avg", "block"]:
                desc_seg_list = search_dict["seg"]
            else:
                desc_seg_list = search_dict["desc"]
            for desc_seg in desc_seg_list:
                if num_layers == 2:
                    alpha_list = search_dict["alpha"]
                else:
                    alpha_list = [
                        f"{alpha2}_{alpha3}"
                        for alpha2, alpha3 in product(
                            search_dict["alpha"], search_dict["alpha"]
                        )
                    ]
                for alpha in alpha_list:
                    ### At this point, all unique vals
                    res_dict = {"diff_T_S": []}
                    for perm_iter, perm_df in r2_df_dict.items():
                        condition = (
                            (perm_df["agg"] == agg)
                            & (perm_df["llama_model"] == llama_model)
                            & (perm_df["desc_seg"] == desc_seg)
                            & (perm_df["alpha"] == alpha)
                        )
                        cur_row = perm_df[condition]
                        r2_S = cur_row["r2_S"].values[0]
                        r2_Q = cur_row["r2_Q"].values[0]
                        r2_T = cur_row["r2_T"].values[0]
                        diff = r2_T - r2_S

                        if perm_iter == "baseline":
                            # Save baseline diff seperately
                            res_dict["baseline"] = diff
                        # Append perm diffs and baseline diff
                        res_dict["diff_T_S"].append(diff)
                    ## P-VALUE
                    # % of perms as or more extreme than baseline (in pos direction)
                    extreme = len(
                        [
                            1
                            for perm_result_diff in res_dict["diff_T_S"]
                            if perm_result_diff >= res_dict["baseline"]
                        ]
                    )
                    p_val = extreme / len(res_dict["diff_T_S"])
                    p_val_list.append(
                        {
                            "agg": agg,
                            "llama_model": llama_model,
                            "desc_seg": desc_seg,
                            "alpha": alpha,
                            "baseline_diff": round(res_dict["baseline"], 4),
                            "p_val": p_val,
                        }
                    )
    return pd.DataFrame(p_val_list)

In [ ]:
p_parent_2 = get_p_df(
    search_dict=search_dict, num_layers=2, r2_df_dict=parent_2_df_dict
)
p_parent_3 = get_p_df(
    search_dict=search_dict, num_layers=3, r2_df_dict=parent_3_df_dict
)
p_self_2 = get_p_df(search_dict=search_dict, num_layers=2, r2_df_dict=self_2_df_dict)
p_self_3 = get_p_df(search_dict=search_dict, num_layers=3, r2_df_dict=self_3_df_dict)

In [ ]:
p_parent_3.sort_values(by="p_val", ascending=True).head(30)

In [ ]:
def get_row(df):
    condition = (
        (df["agg"] == "0_shot_binary")
        & (df["llama_model"] == "Llama-3.3-8B-Instruct")
        & (df["desc_seg"] == "desc_augment")
        & (df["alpha"] == "1.0_1.0")
    )
    return df[condition]

In [ ]:
parent_3_df_dict.keys()

In [ ]:
diff_list = []
for perm_name, perm_df in parent_3_df_dict.items():
    sub_df = get_row(perm_df)
    r2_T = float(sub_df["r2_T"].values[0])
    r2_S = float(sub_df["r2_S"].values[0])
    diff = r2_T - r2_S
    diff_list.append(diff)
    if perm_name == "baseline":
        print(diff)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(diff_list, bins=20)